In [ ]:
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(plyr))
suppressPackageStartupMessages(library(dplyr))

# 1. Load the DeepSEM Output
work.dir <- '/sdata/activities/white-morphic/users/whitebr/kayee-mini-challenge/DeepSEM_Results/'
deepsem.file <- paste0(work.dir, "GRN_inference_result.tsv") # Adjust if DeepSEM names it differently
all.res <- as.data.frame(fread(deepsem.file))

# Standardize column names to match the GRNBoost2 pipeline
# (Ensure these match what DeepSEM actually outputs)
colnames(all.res)[colnames(all.res) == "weight"] <- "score"
colnames(all.res)[colnames(all.res) == "cell_type"] <- "cell.type"

# 2. Global Scaling [-1, 1]
# We skip the correlation step because DeepSEM's 'score' is already natively signed!
# We just scale it globally by the absolute maximum, exactly like GRNBoost2.
all.res[,"score"] <- all.res[,"score"] / max(abs(all.res[,"score"]))

# 3. Add p-values and 4-Sigma Significance
pval_cutoff = pnorm(4, lower.tail=FALSE)

all.res <- ddply(all.res, .variables = c("cell.type", "TF"),
        .fun = function(df) {
          cols <- colnames(df)
          cols <- cols[!(cols %in% c("cell.type", "TF"))]

          # Calculate p-value centered at 0 using the DeepSEM weights
          df[,"pval"] <- pnorm(df$score, mean = 0, sd = sd(df$score), lower.tail=FALSE)
          df[,"padj"] <- p.adjust(df[,"pval"])
          df[,"significant"] <- FALSE

          flag <- df[,"pval"] < pval_cutoff
          df[flag, "significant"] <- TRUE
          df[, c(cols, "pval", "padj", "significant")]
        })

# 4. Save the cleanly formatted output for VIPER!
odir <- paste0(work.dir, "resources/")
dir.create(odir, showWarnings=FALSE)
ofile <- paste0(odir, "postprocessed-deepsem-all-tfs-celltypes.csv.gz")

print(paste("Saving to:", ofile))
fwrite(as.data.frame(all.res), file = ofile, row.names=FALSE, col.names=TRUE, quote=FALSE, sep=",")